# T07 — Training UNet Baseline su LOL-v2 Real

Addestra `UNetBaseline` (base_channels=32, ~7.8M parametri) sulla partizione
**Real_captured** di LOL-v2.

| | |
|---|---|
| Dataset | LOL-v2 Real_captured |
| Train | 90% dei 689 sample ufficiali (≈620 immagini) |
| Val | 10% dei 689 sample ufficiali (≈69 immagini) |
| Test | 100 sample ufficiali (cartella Test — mai visti durante il training) |
| Hardware target | Kaggle T4 (16 GB VRAM) |
| Checkpoint output | `checkpoints/baseline/best.pt` e `last.pt` |

**Come usare su Kaggle:**
1. Carica la repo come dataset (`Add Data → Your datasets`).
2. Carica il dataset LOL-v2 (`Add Data`).
3. Aggiusta `LOLV2_ROOT` nella cella **Costanti** con il path corretto sotto `/kaggle/input/`.
4. Esegui tutte le celle (`Run All`).
5. Scarica `best.pt` dall'output Kaggle.

In [ ]:
# ── Installazione dipendenze mancanti (necessario su Kaggle) ──────────────
import subprocess, sys

def _pip(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", *pkgs, "-q"])

try:
    import piq
except ImportError:
    _pip("piq>=0.8.0")

try:
    import torchinfo
except ImportError:
    _pip("torchinfo>=1.8.0")

print("Dipendenze OK")

In [ ]:
# ── Import ────────────────────────────────────────────────────────────────
import os
import sys
from pathlib import Path

import torch
from torch.utils.data import DataLoader, Subset

# ── Rilevamento ambiente ──────────────────────────────────────────────────
ON_KAGGLE = Path("/kaggle/working").exists()

if ON_KAGGLE:
    # Su Kaggle il progetto è montato come dataset; aggiusta il nome se necessario
    PROJECT_ROOT = Path("/kaggle/input/low-light-enhancement")
    OUTPUT_ROOT  = Path("/kaggle/working")
else:
    # In locale i notebook sono in notebooks/, la root è un livello su
    PROJECT_ROOT = Path("..").resolve()
    OUTPUT_ROOT  = PROJECT_ROOT

# Aggiunge la root al PYTHONPATH per importare src.*
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"OUTPUT_ROOT  : {OUTPUT_ROOT}")
print(f"ON_KAGGLE    : {ON_KAGGLE}")
print(f"CUDA         : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU          : {torch.cuda.get_device_name(0)}")
    print(f"VRAM         : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ── Costanti — modifica LOLV2_ROOT se necessario ──────────────────────────

if ON_KAGGLE:
    # ⚠️  Aggiusta con il nome del tuo dataset Kaggle
    LOLV2_ROOT = Path("/kaggle/input/lol-dataset/LOL-v2")
else:
    LOLV2_ROOT = PROJECT_ROOT / "data" / "LOL-v2"

# Percorsi cartelle LOL-v2 Real_captured
TRAIN_LOW    = LOLV2_ROOT / "Real_captured" / "Train" / "Low"
TRAIN_NORMAL = LOLV2_ROOT / "Real_captured" / "Train" / "Normal"
TEST_LOW     = LOLV2_ROOT / "Real_captured" / "Test"  / "Low"
TEST_NORMAL  = LOLV2_ROOT / "Real_captured" / "Test"  / "Normal"

# Cartelle output
SPLITS_DIR   = OUTPUT_ROOT / "data" / "splits"
CKPT_DIR     = OUTPUT_ROOT / "checkpoints"
LOG_DIR      = OUTPUT_ROOT / "runs"

# Iperparametri training
BATCH_SIZE   = 8      # 256×256 × 8 ≈ 3 GB → sicuro su T4 con AMP
NUM_WORKERS  = 2      # Kaggle T4 ha 2 CPU cores allocati
IMG_SIZE     = 256

# Verifica percorsi
for p in [TRAIN_LOW, TRAIN_NORMAL, TEST_LOW, TEST_NORMAL]:
    status = "✓" if p.exists() else "✗ NON TROVATO"
    print(f"  {status}  {p}")

In [ ]:
# ── Split deterministici train / val / test ───────────────────────────────
#
# LOL-v2 Real fornisce una cartella Train ufficiale (689 immagini) e una
# cartella Test ufficiale (100 immagini). Partiamo dalla Train ufficiale
# e la dividiamo in 90% train / 10% val con seed fisso.
# Il Test ufficiale rimane intatto come held-out set per la valutazione.

from src.data.dataset import PairedImageDataset
from src.data.splits  import create_and_save_train_val_split, save_test_split

# key_fn per LOL-v2 Real: "low00001" ↔ "normal00001" → chiave "00001"
lolv2_key = lambda s: s.lstrip("abcdefghijklmnopqrstuvwxyz")

# Dataset completo della cartella Train ufficiale (solo per raccogliere gli stem)
full_train_ds = PairedImageDataset(TRAIN_LOW, TRAIN_NORMAL, key_fn=lolv2_key)
test_ds_raw   = PairedImageDataset(TEST_LOW,  TEST_NORMAL,  key_fn=lolv2_key)

print(f"Train ufficiale: {len(full_train_ds)} coppie")
print(f"Test ufficiale : {len(test_ds_raw)} coppie")

# Crea e salva gli split (riproducibili: seed=42)
SPLITS_DIR.mkdir(parents=True, exist_ok=True)
splits = create_and_save_train_val_split(
    full_train_ds.stems,
    splits_dir=SPLITS_DIR,
    split_name="lolv2_real",
    val_fraction=0.10,
    seed=42,
)
save_test_split(test_ds_raw.stems, SPLITS_DIR / "lolv2_real_test.txt")

train_stems = splits["train"]
val_stems   = splits["val"]

print(f"\nSplit salvati in: {SPLITS_DIR}")

In [ ]:
# ── Dataset e DataLoader ──────────────────────────────────────────────────
from src.data.transforms import get_preprocessing_transform, get_paired_augmentation

preprocess = get_preprocessing_transform(IMG_SIZE)
augment    = get_paired_augmentation(IMG_SIZE)       # solo per training

# Dataset completo con augmentation (training)
train_ds_full = PairedImageDataset(
    TRAIN_LOW, TRAIN_NORMAL,
    key_fn=lolv2_key,
    paired_transform=augment,
    transform=preprocess,
)

# Dataset completo senza augmentation (val e test)
eval_ds_full = PairedImageDataset(
    TRAIN_LOW, TRAIN_NORMAL,
    key_fn=lolv2_key,
    transform=preprocess,
)

# Costruisce gli indici degli split
train_set = set(train_stems)
val_set   = set(val_stems)
train_indices = [i for i, s in enumerate(train_ds_full.stems) if s in train_set]
val_indices   = [i for i, s in enumerate(eval_ds_full.stems)  if s in val_set]

train_subset = Subset(train_ds_full, train_indices)
val_subset   = Subset(eval_ds_full,  val_indices)

# DataLoader
train_loader = DataLoader(
    train_subset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=(NUM_WORKERS > 0),
    drop_last=True,          # evita batch piccoli a fine epoca
)
val_loader = DataLoader(
    val_subset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=(NUM_WORKERS > 0),
)

print(f"Train samples : {len(train_subset)}  ({len(train_loader)} batch × {BATCH_SIZE})")
print(f"Val samples   : {len(val_subset)}  ({len(val_loader)} batch × {BATCH_SIZE})")

# Sanity check — verifica shape di un batch
batch = next(iter(train_loader))
print(f"\nBatch shape: low={tuple(batch['low'].shape)}  normal={tuple(batch['normal'].shape)}")
print(f"Range: [{batch['low'].min():.3f}, {batch['low'].max():.3f}]")

In [ ]:
# ── Modello, Loss e TrainConfig ───────────────────────────────────────────
from src.models  import UNetBaseline
from src.losses  import CombinedLoss
from src.train   import TrainConfig, Trainer

model     = UNetBaseline(base_channels=32)
criterion = CombinedLoss(alpha=0.8)   # 0.8 × L1 + 0.2 × (1 - SSIM)

config = TrainConfig(
    # ── Ottimizzatore ────────────────────────────────────────────────────
    optimizer_name   = "adamw",
    lr               = 1e-4,
    weight_decay     = 1e-4,
    grad_clip_norm   = 1.0,

    # ── Schedule ─────────────────────────────────────────────────────────
    epochs               = 100,
    scheduler_name       = "cosine",
    early_stopping_patience = 15,

    # ── Validazione e logging ─────────────────────────────────────────────
    val_every_n_epochs  = 1,
    log_every_n_epochs  = 1,
    n_visual_samples    = 4,

    # ── Hardware e riproducibilità ────────────────────────────────────────
    amp    = True,     # fp16 su T4; ignorato automaticamente se device=cpu
    device = "auto",   # cuda se disponibile, altrimenti cpu
    seed   = 42,

    # ── Checkpoint e log ─────────────────────────────────────────────────
    checkpoint_dir   = str(CKPT_DIR),
    log_dir          = str(LOG_DIR),
    experiment_name  = "baseline",
)

print(model)
print(f"\nParametri: {sum(p.numel() for p in model.parameters()):,}")
print(f"\nConfig:")
for field, value in vars(config).items():
    print(f"  {field:<28} = {value}")

In [ ]:
# ── Training ──────────────────────────────────────────────────────────────
# trainer.fit() esegue:
#   - train_epoch(): forward + backward + grad_clip + optimizer_step
#   - val_epoch(): loss + PSNR + SSIM su val_loader
#   - _log_visuals(): griglia low|output|gt su TensorBoard
#   - early stopping con patience=15
#   - salva best.pt (miglior val_loss) e last.pt ad ogni epoca

trainer = Trainer(model, train_loader, val_loader, criterion, config)
trainer.fit()

In [ ]:
# ── Riepilogo finale ──────────────────────────────────────────────────────
import json
from dataclasses import asdict

best_ckpt = trainer.ckpt_dir / "best.pt"
last_ckpt = trainer.ckpt_dir / "last.pt"

print("=" * 60)
print("RIEPILOGO TRAINING")
print("=" * 60)
print(f"Epoche completate : {trainer.epoch}")
print(f"Best val_loss     : {trainer.best_score:.6f}")
print(f"Checkpoint best   : {best_ckpt}  ({best_ckpt.stat().st_size / 1e6:.1f} MB)")
print(f"Checkpoint last   : {last_ckpt}")
print()

# Valutazione finale sul val set con il best checkpoint
ckpt = torch.load(best_ckpt, map_location=trainer.device, weights_only=False)
trainer.model.load_state_dict(ckpt["model"])
final_metrics = trainer.val_epoch()

print("Metriche val (best checkpoint):")
print(f"  val_loss : {final_metrics['loss']:.6f}")
print(f"  PSNR     : {final_metrics['psnr']:.2f} dB")
print(f"  SSIM     : {final_metrics['ssim']:.4f}")
print()

# Salva un file di riepilogo leggibile accanto al checkpoint
summary = {
    "experiment"    : config.experiment_name,
    "model"         : str(trainer.model),
    "epochs_done"   : trainer.epoch,
    "best_val_loss" : trainer.best_score,
    "val_psnr_db"   : final_metrics["psnr"],
    "val_ssim"      : final_metrics["ssim"],
    "hardware"      : torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
    "config"        : asdict(config),
}
(trainer.ckpt_dir / "training_summary.json").write_text(
    json.dumps(summary, indent=2), encoding="utf-8"
)
print(f"Summary salvato in: {trainer.ckpt_dir / 'training_summary.json'}")

# Su Kaggle lista i file da scaricare
if ON_KAGGLE:
    import os
    print("\nFile da scaricare dall'output Kaggle:")
    for f in sorted(trainer.ckpt_dir.iterdir()):
        print(f"  {f.relative_to(OUTPUT_ROOT)}  ({f.stat().st_size / 1e6:.1f} MB)")